In [143]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeClassifier
from sklearn.metrics import accuracy_score, f1_score

# Dataset

In [46]:
data_origin_cate = pd.read_excel("./widsdatathon2025/TRAIN_NEW/TRAIN_CATEGORICAL_METADATA_new.xlsx")

data_origin_quan = pd.read_excel("./widsdatathon2025/TRAIN_NEW/TRAIN_QUANTITATIVE_METADATA_new.xlsx")

data_origin_funct_matrix = pd.read_csv("./widsdatathon2025/TRAIN_NEW/TRAIN_FUNCTIONAL_CONNECTOME_MATRICES_new_36P_Pearson.csv")

data_origin_solution = pd.read_excel("./widsdatathon2025/TRAIN_NEW/TRAINING_SOLUTIONS.xlsx")

data_dict = pd.read_excel("./widsdatathon2025/Data Dictionary.xlsx")

data_test_cate = pd.read_excel("./widsdatathon2025/TEST/TEST_CATEGORICAL.xlsx")

data_test_quan = pd.read_excel("./widsdatathon2025/TEST/TEST_QUANTITATIVE_METADATA.xlsx")

data_test_funct_matrix = pd.read_csv("./widsdatathon2025/TEST/TEST_FUNCTIONAL_CONNECTOME_MATRICES.csv")

In [91]:
data_training_funct_matrix = data_origin_funct_matrix
data_training_solution = data_origin_solution

In [92]:
final_data_cat = pd.merge(data_origin_cate, data_origin_quan, on = 'participant_id')
final_data_mat = pd.merge(final_data_cat, data_origin_funct_matrix, on = 'participant_id')
final_data = pd.merge(final_data_mat, data_origin_solution, on = 'participant_id')
final_data

,participant_id,Basic_Demos_Enroll_Year,Basic_Demos_Study_Site,PreInt_Demos_Fam_Child_Ethnicity,PreInt_Demos_Fam_Child_Race,MRI_Track_Scan_Location,Barratt_Barratt_P1_Edu,Barratt_Barratt_P1_Occ,Barratt_Barratt_P2_Edu,Barratt_Barratt_P2_Occ,...,195throw_198thcolumn,195throw_199thcolumn,196throw_197thcolumn,196throw_198thcolumn,196throw_199thcolumn,197throw_198thcolumn,197throw_199thcolumn,198throw_199thcolumn,ADHD_Outcome,Sex_F
0,00aIpNTbG5uh,2019,4,1.0,0.0,3.0,21.0,45.0,NaN,NaN,...,-0.280312,0.037560,0.423037,0.242453,0.336213,0.402338,0.327915,0.539032,1,0
1,00fV0OyyoLfw,2017,1,0.0,9.0,2.0,21.0,0.0,21.0,45.0,...,-0.332783,-0.332711,0.556939,0.475578,0.429196,0.457970,0.312571,0.595978,1,0
2,04X1eiS79T4B,2017,1,1.0,2.0,2.0,9.0,0.0,NaN,NaN,...,-0.002132,-0.175586,0.679183,0.290292,0.486680,0.255208,0.575017,0.605182,0,1
3,05ocQutkURd6,2018,1,3.0,8.0,2.0,18.0,10.0,18.0,0.0,...,-0.199576,-0.216457,0.519074,0.298586,0.415466,0.511607,0.361204,0.446613,0,1
4,06YUNBA9ZRLq,2018,1,0.0,1.0,2.0,12.0,0.0,NaN,NaN,...,-0.141012,-0.002865,0.515169,0.336139,0.316430,0.442230,0.177079,0.378278,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1208,zwjJWCRzKhDz,2019,4,1.0,1.0,3.0,12.0,NaN,15.0,5.0,...,0.112789,0.211312,0.601190,0.587116,0.312695,0.485938,0.189102,0.354333,0,1
1209,zwXD5v17Rx01,2018,1,0.0,0.0,3.0,21.0,40.0,21.0,40.0,...,0.253990,0.198741,0.648260,0.055241,0.491985,0.118676,0.404331,0.537121,1,0
1210,zWzLCi3NTBTd,2018,3,2.0,3.0,3.0,21.0,40.0,21.0,35.0,...,0.044653,0.234887,0.538475,0.024265,0.472322,0.095624,0.205326,0.182633,1,1
1211,Zy9GTHDxUbXU,2019,4,0.0,1.0,3.0,18.0,35.0,18.0,45.0,...,-0.035955,-0.062152,0.706214,0.183288,0.104987,0.420463,0.152727,0.706737,1,0


In [93]:
final_data_test = pd.merge(data_test_quan, data_test_funct_matrix, on = 'participant_id')
print(len(final_data_test.columns))
final_data = pd.merge(data_origin_quan, data_origin_funct_matrix, on = 'participant_id')
final_data = pd.merge(final_data, data_origin_solution, on = 'participant_id')
len(final_data.columns)

19919


19921

In [144]:
df = pd.DataFrame(data = {'Percent_missing' : final_data.isnull().sum() * 100 / len(final_data)}, copy = False)
df[df['Percent_missing'] > 0]

,Percent_missing
EHQ_EHQ_Total,1.071723
ColorVision_CV_Score,1.896125
APQ_P_APQ_P_CP,0.989283
APQ_P_APQ_P_ID,0.989283
APQ_P_APQ_P_INV,0.989283
APQ_P_APQ_P_OPD,0.989283
APQ_P_APQ_P_PM,0.989283
APQ_P_APQ_P_PP,0.989283
SDQ_SDQ_Conduct_Problems,0.741962
SDQ_SDQ_Difficulties_Total,0.741962


# Train Model from Hoang

In [95]:
matrix_filtered = pd.read_csv('./Thuỷ Đức/matrix_filter_feature.csv', index_col=0)
matrix_list_index = matrix_filtered['index'].to_list()

In [52]:
matrix_filtered

,index,F-score,p-value
0,131throw_185thcolumn,39.010944,5.822705e-10
1,152throw_184thcolumn,33.786638,7.857024e-09
2,131throw_189thcolumn,32.468953,1.519578e-08
3,57throw_189thcolumn,30.669050,3.749940e-08
4,164throw_189thcolumn,29.588230,6.459198e-08
...,...,...,...
1405,17throw_51thcolumn,6.666266,9.942098e-03
1406,57throw_150thcolumn,6.662546,9.962756e-03
1407,171throw_180thcolumn,6.662375,9.963705e-03
1408,120throw_131thcolumn,6.661555,9.968265e-03


In [96]:
from sklearn.feature_selection import SelectKBest, f_classif

data = pd.merge(data_training_funct_matrix, data_training_solution, on = 'participant_id')
columns = data.columns
len_column = len(data.columns)
x = data[columns[1:len_column - 2]]
y = data['Sex_F']
selector = SelectKBest(f_classif, k = len(columns[1:len_column - 2]))
selector.fit(x, y)

final = pd.DataFrame(data = {'F-score': selector.scores_, 'p-value': selector.pvalues_}, index = columns[1:len_column - 2]).sort_values(by = 'F-score', ascending = False)
final_sort = final.sort_values(by = 'p-value', ascending = True)
#final_sort[final_sort['p-value'] <= 0.00005].reset_index()

matrix_sex_filtered = final_sort[final_sort['p-value'] <= 0.005].reset_index()
matrix_sex_list = matrix_sex_filtered['index'].tolist()
len(matrix_sex_list)

978

In [97]:
from sklearn.feature_selection import SelectKBest, f_classif

data = pd.merge(data_training_funct_matrix, data_training_solution, on = 'participant_id')
columns = data.columns
len_column = len(data.columns)
x = data[columns[1:len_column - 2]]
y = data['ADHD_Outcome']
selector = SelectKBest(f_classif, k = len(columns[1:len_column - 2]))
selector.fit(x, y)

final = pd.DataFrame(data = {'F-score': selector.scores_, 'p-value': selector.pvalues_}, index = columns[1:len_column - 2]).sort_values(by = 'F-score', ascending = False)
final_sort = final.sort_values(by = 'p-value', ascending = True)
final_sort[final_sort['p-value'] <= 0.0005].reset_index()

matrix_adhd_filtered = final_sort[final_sort['p-value'] <= 0.0005].reset_index()
matrix_adhd_list = matrix_adhd_filtered['index'].to_list()
matrix_adhd_list

['166throw_184thcolumn',
 '78throw_170thcolumn',
 '1throw_16thcolumn',
 '2throw_166thcolumn',
 '11throw_166thcolumn',
 '78throw_189thcolumn',
 '58throw_163thcolumn',
 '76throw_170thcolumn',
 '105throw_166thcolumn',
 '53throw_127thcolumn',
 '50throw_197thcolumn',
 '159throw_166thcolumn',
 '0throw_166thcolumn',
 '50throw_189thcolumn',
 '34throw_130thcolumn',
 '5throw_166thcolumn',
 '98throw_162thcolumn',
 '24throw_162thcolumn',
 '143throw_199thcolumn',
 '96throw_169thcolumn',
 '50throw_192thcolumn',
 '0throw_58thcolumn',
 '123throw_143thcolumn',
 '73throw_185thcolumn',
 '104throw_166thcolumn',
 '166throw_174thcolumn',
 '76throw_189thcolumn',
 '87throw_159thcolumn',
 '96throw_170thcolumn',
 '108throw_144thcolumn']

In [98]:
features_adhd = ['SDQ_SDQ_Hyperactivity', 'SDQ_SDQ_Externalizing', 
                 'SDQ_SDQ_Difficulties_Total', 'SDQ_SDQ_Generating_Impact', 
                 'SDQ_SDQ_Conduct_Problems']

features_adhd + matrix_adhd_list

['SDQ_SDQ_Hyperactivity',
 'SDQ_SDQ_Externalizing',
 'SDQ_SDQ_Difficulties_Total',
 'SDQ_SDQ_Generating_Impact',
 'SDQ_SDQ_Conduct_Problems',
 '166throw_184thcolumn',
 '78throw_170thcolumn',
 '1throw_16thcolumn',
 '2throw_166thcolumn',
 '11throw_166thcolumn',
 '78throw_189thcolumn',
 '58throw_163thcolumn',
 '76throw_170thcolumn',
 '105throw_166thcolumn',
 '53throw_127thcolumn',
 '50throw_197thcolumn',
 '159throw_166thcolumn',
 '0throw_166thcolumn',
 '50throw_189thcolumn',
 '34throw_130thcolumn',
 '5throw_166thcolumn',
 '98throw_162thcolumn',
 '24throw_162thcolumn',
 '143throw_199thcolumn',
 '96throw_169thcolumn',
 '50throw_192thcolumn',
 '0throw_58thcolumn',
 '123throw_143thcolumn',
 '73throw_185thcolumn',
 '104throw_166thcolumn',
 '166throw_174thcolumn',
 '76throw_189thcolumn',
 '87throw_159thcolumn',
 '96throw_170thcolumn',
 '108throw_144thcolumn']

In [ ]:
# features_adhd = ['SDQ_SDQ_Hyperactivity', 'SDQ_SDQ_Externalizing', 
#                  'SDQ_SDQ_Difficulties_Total', 'SDQ_SDQ_Generating_Impact', 
#                  'SDQ_SDQ_Conduct_Problems']
# features_sex_f = ['SDQ_SDQ_Hyperactivity', 'SDQ_SDQ_Externalizing', 
#                   'SDQ_SDQ_Prosocial', 'SDQ_SDQ_Emotional_Problems', 
#                   'ColorVision_CV_Score']

# features_sex_f = features_sex_f + matrix_sex_list
# features_adhd = features_adhd + matrix_adhd_list

# label_adhd = 'ADHD_Outcome'
# label_sex_f = 'Sex_F'

# X_adhd = final_data[features_adhd]
# X_test_adhd = final_data_test[features_adhd]

# #X_sex_f = final_data.drop(columns=['participant_id', 'ADHD_Outcome', 'Sex_F'])

# X_sex_f = final_data[features_sex_f]
# # X_test_sex = final_data_test.drop(columns=['participant_id'])
# X_test_sex = final_data_test[features_sex_f]

# X_adhd = X_adhd.fillna(X_adhd.median())
# X_test_adhd = X_test_adhd.fillna(X_test_adhd.median())
# X_sex_f = X_sex_f.fillna(X_sex_f.median())
# X_test_sex = X_test_sex.fillna(X_test_sex.median())

# # X_adhd_train, X_adhd_test, y_adhd_train, y_adhd_test = train_test_split(X_adhd, y_adhd, test_size=0.2, random_state=42)

# # X_sex_f = final_data[features_sex_f]
# y_adhd_train = final_data[label_adhd]
# y_sex_f_train = final_data[label_sex_f]

# # X_sex_f_train, X_sex_f_test, y_sex_f_train, y_sex_f_test = train_test_split(X_sex_f, y_sex_f, test_size=0.2, random_state=42)

# # Standardize the features
# scaler = StandardScaler()
# X_adhd_train_scaled = scaler.fit_transform(X_adhd)
# X_test_scaled_adhd = scaler.transform(X_test_adhd)
# X_sex_f_train_scaled = scaler.fit_transform(X_sex_f)
# X_test_scaled_sex = scaler.transform(X_test_sex)


# # Train RidgeClassifier for ADHD_Outcome
# ridge_adhd = RidgeClassifier(alpha=1.0)  # Adjust alpha for regularization strength
# ridge_adhd.fit(X_adhd_train_scaled, y_adhd_train)
# y_adhd_pred_train = ridge_adhd.predict(X_adhd_train_scaled)
# y_adhd_pred_test = ridge_adhd.predict(X_test_scaled_adhd)

# # Train RidgeClassifier for Sex_F
# ridge_sex_f = RidgeClassifier(alpha=1.0)
# ridge_sex_f.fit(X_sex_f_train_scaled, y_sex_f_train)
# y_sex_f_pred_train = ridge_sex_f.predict(X_sex_f_train_scaled)
# y_sex_f_pred_test = ridge_sex_f.predict(X_test_scaled_sex)

# # Compute F1-score and accuracy
# f1_adhd_train = f1_score(y_adhd_train, y_adhd_pred_train)
# f1_sex_f_train = f1_score(y_sex_f_train, y_sex_f_pred_train)



# # Print results
# print(f"ADHD_Outcome - F1 Score (Train): {f1_adhd_train}")
# print(f"Sex_F - F1 Score (Train): {f1_sex_f_train}")



# # Save predictions to CSV
# submission = final_data_test[['participant_id']].copy()
# submission['ADHD_Outcome'] = y_adhd_pred_test  # Assuming the first column corresponds to ADHD_Outcome
# submission['Sex_F'] = y_sex_f_pred_test

# # Save results to CSV
# submission.to_csv("submission-1.csv", index=False)
# print("Predictions saved to submission.csv")


ADHD_Outcome - F1 Score (Train): 0.8815331010452961
Sex_F - F1 Score (Train): 1.0
Predictions saved to submission.csv


# ThuyDuc

## Adjustment for connectome matrix

### Calculate R^2 for connectome matrix

In [145]:
participant = data_origin_funct_matrix['participant_id']
matrix = data_origin_funct_matrix.iloc[:, 1:]
matrix = matrix**2
matrix_adjusted = pd.concat([participant, matrix], axis = 1)
matrix_adjusted

,participant_id,0throw_1thcolumn,0throw_2thcolumn,0throw_3thcolumn,0throw_4thcolumn,0throw_5thcolumn,0throw_6thcolumn,0throw_7thcolumn,0throw_8thcolumn,0throw_9thcolumn,...,195throw_196thcolumn,195throw_197thcolumn,195throw_198thcolumn,195throw_199thcolumn,196throw_197thcolumn,196throw_198thcolumn,196throw_199thcolumn,197throw_198thcolumn,197throw_199thcolumn,198throw_199thcolumn
0,70z8Q2xdTXM3,0.049698,0.278681,0.184871,0.003655,0.320910,0.099441,0.258479,0.006129,0.276352,...,0.050618,0.157965,1.789005e-01,0.034093,0.093360,0.176694,0.000267,0.315691,0.222001,0.133386
1,WHWymJu6zNZi,0.377936,0.333224,0.246142,0.246618,0.163770,0.193357,0.015028,0.007302,0.014562,...,0.047326,0.000212,1.939562e-07,0.009303,0.206571,0.118278,0.027993,0.369246,0.303186,0.253186
2,4PAQp1M6EyAo,0.013650,0.210138,0.067966,0.408361,0.591880,0.195831,0.405909,0.036868,0.270795,...,0.117297,0.000447,1.431571e-03,0.005635,0.170331,0.085678,0.152885,0.213023,0.258991,0.389665
3,obEacy4Of68I,0.039875,0.566579,0.433336,0.330736,0.480065,0.417044,0.273268,0.169899,0.281795,...,0.010725,0.031795,4.451395e-02,0.000348,0.190369,0.351627,0.046744,0.116467,0.193875,0.311580
4,s7WzzDcmDOhF,0.051675,0.376098,0.386196,0.316601,0.542741,0.347879,0.071116,0.129361,0.090463,...,0.027211,0.000050,1.461771e-02,0.238236,0.243617,0.046380,0.044388,0.003119,0.014176,0.011723
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1208,9gpepMI9sj5q,0.070375,0.304402,0.394783,0.419516,0.505176,0.017512,0.106411,0.125542,0.282141,...,0.016154,0.016845,8.939327e-02,0.013128,0.285273,0.014136,0.032867,0.032781,0.056814,0.332940
1209,FIDen5rdMc0v,0.000338,0.332570,0.278205,0.107232,0.344414,0.329119,0.090327,0.090663,0.435389,...,0.232531,0.025468,2.640547e-02,0.008695,0.096046,0.020684,0.047671,0.151579,0.108071,0.056855
1210,dlsMC4TXL4e8,0.051542,0.164559,0.000554,0.008665,0.004756,0.419352,0.581485,0.216326,0.039735,...,0.143872,0.016123,3.751756e-02,0.046003,0.184759,0.091289,0.010978,0.733378,0.091959,0.132233
1211,syeyZjEx8FUx,0.036042,0.566823,0.709743,0.667550,0.672722,0.628769,0.423709,0.238636,0.336640,...,0.045052,0.003085,4.925528e-02,0.040992,0.363188,0.232387,0.159491,0.139686,0.078344,0.468450


### Extract connectome matrix for Sex_F to examine

In [146]:
matrix_adjusted_merge_adhd_sex_f = pd.merge(matrix_adjusted, data_origin_solution, on = 'participant_id')

columns = matrix_adjusted_merge_adhd_sex_f.columns

len_column = len(columns)

columns_for_sex_f = list(columns[:len_column - 2]) + ['Sex_F']

data = matrix_adjusted_merge_adhd_sex_f[columns_for_sex_f]

# Connectome matrix for male and female
data_male = data[data['Sex_F'] == 0]
data_female = data[data['Sex_F'] == 1]

data_male.shape

(797, 19902)

In [147]:
from sklearn.feature_selection import SelectKBest, f_classif

columns = data.columns
len_column = len(columns)

x = data[columns[1:len_column - 1]]
y = data['Sex_F']

selector = SelectKBest(f_classif, k = len(columns[1:len_column - 1]))
selector.fit(x, y)

final = pd.DataFrame(data = {'F-score': selector.scores_, 'p-value': selector.pvalues_}, index = columns[1:len_column - 1])
final_sort = final.sort_values(by = 'p-value', ascending = True)
final_sort[final_sort['p-value'] <= 0.00005]

,F-score,p-value
152throw_184thcolumn,33.219743,1.043343e-08
158throw_191thcolumn,27.810005,1.583863e-07
75throw_185thcolumn,26.363041,3.293567e-07
89throw_91thcolumn,25.846923,4.278574e-07
80throw_185thcolumn,24.931596,6.809667e-07
172throw_188thcolumn,23.990932,1.098897e-06
75throw_80thcolumn,23.595797,1.343969e-06
184throw_185thcolumn,21.370720,4.190635e-06
60throw_75thcolumn,21.172609,4.638609e-06
32throw_87thcolumn,21.129015,4.743476e-06


In [173]:
# Extract the features with p-value <= 0.0005
connectome_features_sex = final_sort[final_sort['p-value'] <= 0.00002].index
len(connectome_features_sex)

18

### Extract connectome matrix for ADHD to examine

In [149]:
matrix_adjusted_merge_adhd_sex_f = pd.merge(matrix_adjusted, data_origin_solution, on = 'participant_id')

columns = matrix_adjusted_merge_adhd_sex_f.columns

len_column = len(columns)

columns_for_ADHD_outcome = list(columns[:len_column - 2]) + ['ADHD_Outcome']

data = matrix_adjusted_merge_adhd_sex_f[columns_for_ADHD_outcome]

# Connectome matrix for male and female
data_ADHD_1 = data[data['ADHD_Outcome'] == 1]
data_ADHD_0 = data[data['ADHD_Outcome'] == 0]

data_ADHD_1.head()

,participant_id,0throw_1thcolumn,0throw_2thcolumn,0throw_3thcolumn,0throw_4thcolumn,0throw_5thcolumn,0throw_6thcolumn,0throw_7thcolumn,0throw_8thcolumn,0throw_9thcolumn,...,195throw_197thcolumn,195throw_198thcolumn,195throw_199thcolumn,196throw_197thcolumn,196throw_198thcolumn,196throw_199thcolumn,197throw_198thcolumn,197throw_199thcolumn,198throw_199thcolumn,ADHD_Outcome
0,70z8Q2xdTXM3,0.049698,0.278681,0.184871,0.003655,0.320910,0.099441,0.258479,0.006129,0.276352,...,0.157965,1.789005e-01,0.034093,0.093360,0.176694,0.000267,0.315691,0.222001,0.133386,1
1,WHWymJu6zNZi,0.377936,0.333224,0.246142,0.246618,0.163770,0.193357,0.015028,0.007302,0.014562,...,0.000212,1.939562e-07,0.009303,0.206571,0.118278,0.027993,0.369246,0.303186,0.253186,1
2,4PAQp1M6EyAo,0.013650,0.210138,0.067966,0.408361,0.591880,0.195831,0.405909,0.036868,0.270795,...,0.000447,1.431571e-03,0.005635,0.170331,0.085678,0.152885,0.213023,0.258991,0.389665,1
3,obEacy4Of68I,0.039875,0.566579,0.433336,0.330736,0.480065,0.417044,0.273268,0.169899,0.281795,...,0.031795,4.451395e-02,0.000348,0.190369,0.351627,0.046744,0.116467,0.193875,0.311580,1
4,s7WzzDcmDOhF,0.051675,0.376098,0.386196,0.316601,0.542741,0.347879,0.071116,0.129361,0.090463,...,0.000050,1.461771e-02,0.238236,0.243617,0.046380,0.044388,0.003119,0.014176,0.011723,1


In [105]:
from sklearn.feature_selection import SelectKBest, f_classif

data = matrix_adjusted_merge_adhd_sex_f[columns_for_ADHD_outcome]

columns = data.columns

len_column = len(columns)

x = data[columns[1:len_column - 1]]
y = data['ADHD_Outcome']

selector = SelectKBest(f_classif, k = len(columns[1:len_column - 1]))
selector.fit(x, y)

final = pd.DataFrame(data = {'F-score': selector.scores_, 'p-value': selector.pvalues_}, index = columns[1:len_column - 1])
final_sort = final.sort_values(by = 'p-value', ascending = True)
final_sort[final_sort['p-value'] <= 0.0005]

,F-score,p-value
78throw_170thcolumn,18.452153,0.000019
166throw_184thcolumn,16.824730,0.000044
78throw_189thcolumn,14.821830,0.000124
34throw_130thcolumn,14.650053,0.000136
34throw_134thcolumn,14.647255,0.000136
76throw_170thcolumn,14.435530,0.000152
104throw_119thcolumn,14.050577,0.000186
80throw_134thcolumn,13.182342,0.000294
24throw_143thcolumn,13.080956,0.000311
1throw_171thcolumn,12.616961,0.000397


In [64]:
# Extract the features with p-value <= 0.0005
connectome_features_ADHD = final_sort[final_sort['p-value'] <= 0.0005].index
connectome_features_ADHD

Index(['78throw_170thcolumn', '166throw_184thcolumn', '78throw_189thcolumn',
       '34throw_130thcolumn', '34throw_134thcolumn', '76throw_170thcolumn',
       '104throw_119thcolumn', '80throw_134thcolumn', '24throw_143thcolumn',
       '1throw_171thcolumn', '17throw_35thcolumn', '132throw_166thcolumn'],
      dtype='object')

## Train model based on chosen features

'SDQ_SDQ_Emotional_Problems'

In [191]:
# features without connectome matrix
# features_adhd = ['SDQ_SDQ_Hyperactivity', 'SDQ_SDQ_Externalizing', 
#                  'SDQ_SDQ_Difficulties_Total', 'SDQ_SDQ_Generating_Impact', 
#                  'SDQ_SDQ_Conduct_Problems']
features_adhd = ['SDQ_SDQ_Hyperactivity']
# features_sex_f = ['SDQ_SDQ_Hyperactivity', 'SDQ_SDQ_Externalizing', 
#                   'SDQ_SDQ_Prosocial', 
#                   'ColorVision_CV_Score']
features_sex_f = ['SDQ_SDQ_Prosocial']

# features from connectome matrix
# features_connectome_adhd = connectome_features_ADHD.to_list()

features_connectome_sex = connectome_features_sex.to_list()

In [175]:
len(features_connectome_sex)

18

## Models

### Preparation

In [192]:
# Get features and labels
features_sex_f_completed = features_connectome_sex + ['Sex_F']
features_adhd_completed = features_adhd + ['ADHD_Outcome']
#features_connectome_adhd -- used when needed
#features_sex_f -- used when needed

In [193]:
# Get datasets with selected features

# For ADHD
dataset_adhd_train = final_data[features_adhd_completed]
dataset_adhd_test = final_data_test[features_adhd_completed[:-1]] # this dataset does not have label

# For sex_F
dataset_sex_f_train = final_data[features_sex_f_completed]
dataset_sex_f_test = final_data_test[features_sex_f_completed[:-1]] # this dataset does not have label

In [194]:
# Drop missing values
dataset_adhd_train = dataset_adhd_train.dropna()
dataset_adhd_test = dataset_adhd_test.dropna()

dataset_sex_f_train = dataset_sex_f_train.dropna()
dataset_sex_f_test = dataset_sex_f_test.dropna()

In [196]:
# Split data into training and validation dataset
# For ADHD_Outcome
X_train_adhd, X_val_adhd, y_train_adhd, y_val_adhd = train_test_split(dataset_adhd_train.iloc[:, :-1], dataset_adhd_train.iloc[:, -1], test_size=0.2, random_state=42, stratify=dataset_adhd_train.iloc[:, -1])
# For Sex_F
X_train_sex, X_val_sex, y_train_sex, y_val_sex = train_test_split(dataset_sex_f_train.iloc[:, :-1], dataset_sex_f_train.iloc[:, -1], test_size=0.2, random_state=42, stratify=dataset_sex_f_train.iloc[:, -1])

In [198]:
# Normalization
# For ADHD_Outcome
scaler = StandardScaler()
scaler.fit(X_train_adhd)  # Fit on training data

X_train_adhd = scaler.transform(X_train_adhd)  # Transform training data
y_train_adhd = y_train_adhd  # Extract labels

X_test_adhd = scaler.transform(dataset_adhd_test) # Transform test data

X_val_adhd = scaler.transform(X_val_adhd)  # Transform validation data
y_val_adhd = y_val_adhd  # Extract labels




# For Sex_F
# scaler = StandardScaler()
# scaler.fit(X_train_sex)  # Fit on training data

# X_train_sex = scaler.transform(X_train_sex)  # Transform training data
# y_train_sex = y_train_sex.values

# X_test_sex = scaler.transform(dataset_sex_f_test)  # Transform test data

# X_val_sex = scaler.transform(X_val_sex) # Transform validation data
# y_val_sex = y_val_sex.values


d:\Datathon\.venv\Lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


### Comparison between val_datasets and test_datasets

### Test different models

**Create ModelEvaluator to test model**

In [72]:
class ModelEvaluator:
    def __init__(self, models : list, models_name : list[str], X_train_adhd=None, y_train_adhd=None, X_val_adhd=None, y_val_adhd=None, X_train_sex=None, y_train_sex=None, X_val_sex=None, y_val_sex=None):
        self.models = models
        self.models_name = models_name

        self.X_train_adhd = X_train_adhd
        self.y_train_adhd = y_train_adhd
        self.X_val_adhd = X_val_adhd
        self.y_val_adhd = y_val_adhd

        self.X_train_sex = X_train_sex
        self.y_train_sex = y_train_sex
        self.X_val_sex = X_val_sex
        self.y_val_sex = y_val_sex

    def evaluate(self):
        acc_adhd_collect = []
        f1_adhd_collect = []
        acc_sex_collect = []
        f1_sex_collect = []

        # MultiIndex for columns
        columns = pd.MultiIndex.from_tuples([
            ('ADHD_Outcome', 'Accuracy'),
            ('ADHD_Outcome', 'F1 Score'),
            ('Sex F', 'Accuracy'),
            ('Sex F', 'F1 Score')])
        
        df = pd.DataFrame(columns=columns, index=self.models_name)

        for index, model in enumerate(self.models):
            # Evaluate ADHD_Outcome
            model.fit(self.X_train_adhd, self.y_train_adhd)
            y_pred_adhd = model.predict(self.X_val_adhd)

            acc_adhd = accuracy_score(self.y_val_adhd, y_pred_adhd)
            f1_adhd = f1_score(self.y_val_adhd, y_pred_adhd)

            # print(f"{self.models_name[index]} - ADHD_Outcome - Accuracy: {acc_adhd}")
            # print(f"{self.models_name[index]} - ADHD_Outcome - F1 Score: {f1_adhd}")
            
            # Evaluate Sex_F
            model.fit(self.X_train_sex, self.y_train_sex)
            y_pred_sex = model.predict(self.X_val_sex)

            acc_sex = accuracy_score(self.y_val_sex, y_pred_sex)
            f1_sex = f1_score(self.y_val_sex, y_pred_sex)

            # print(f"{self.models_name[index]} - Sex_F - Accuracy: {acc_sex}")
            # print(f"{self.models_name[index]} - Sex_F - F1 Score: {f1_sex}")

            acc_adhd_collect.append(acc_adhd)
            f1_adhd_collect.append(f1_adhd)

            acc_sex_collect.append(acc_sex)
            f1_sex_collect.append(f1_sex)
        
        # Add results to DataFrame
        df['ADHD_Outcome', 'Accuracy'] = acc_adhd_collect
        df['ADHD_Outcome', 'F1 Score'] = f1_adhd_collect
        df['Sex F', 'Accuracy'] = acc_sex_collect
        df['Sex F', 'F1 Score'] = f1_sex_collect

        return df


**Testing different models**

**Trước tiên chưa cần dùng đến các feature từ ma trận connectome cho ADHD_Outcome. Việc có cần dùng hay không phụ thuộc vào việc so sánh phân phối giữa test dataset và train dataset**

**Đối với Sex_F hiện tại chỉ sử dụng ma trận connectome, dùng thêm các feature SDQ nếu cần**

In [73]:
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier, VotingClassifier, HistGradientBoostingClassifier
from sklearn.ensemble import StackingClassifier, AdaBoostClassifier, BaggingClassifier
from sklearn.linear_model import SGDClassifier, Perceptron
from sklearn.svm import SVC, LinearSVC
from sklearn.neighbors import KNeighborsClassifier



ridge = RidgeClassifier(random_state=42, alpha = 0.4)
gaussian = GaussianNB()
decision_tree = DecisionTreeClassifier(random_state=42)
logistic_regression = LogisticRegression(random_state=42, max_iter=1000, C = 0.1)
random_forest = RandomForestClassifier(random_state=42)
extra_trees = ExtraTreesClassifier(random_state=42)
gradient_boosting = GradientBoostingClassifier(random_state=42)
hist_gradient_boosting = HistGradientBoostingClassifier(random_state=42)
sgd = SGDClassifier(random_state=42)
perceptron = Perceptron(random_state=42)
svc = SVC(random_state=42)
linear_svc = LinearSVC(random_state=42)
knn = KNeighborsClassifier()
adaboost = AdaBoostClassifier(random_state=42)
bagging = BaggingClassifier(random_state=42)

stacking = StackingClassifier(estimators=[
    ('ridge', ridge),
    ('gaussian', gaussian),
    ('decision_tree', decision_tree),
    ('logistic_regression', logistic_regression),
    ('random_forest', random_forest),
    ('extra_trees', extra_trees),
    ('gradient_boosting', gradient_boosting),
    ('hist_gradient_boosting', hist_gradient_boosting),
    ('sgd', sgd),
    ('perceptron', perceptron),
    ('svc', svc),
    ('linear_svc', linear_svc),
    ('knn', knn)
], final_estimator=RidgeClassifier(random_state=42))

voting = VotingClassifier(estimators=[
    ('ridge', ridge),
    ('gaussian', gaussian),
    ('decision_tree', decision_tree),
    ('logistic_regression', logistic_regression),
    ('random_forest', random_forest),
    ('extra_trees', extra_trees),
    ('gradient_boosting', gradient_boosting),
    ('hist_gradient_boosting', hist_gradient_boosting),
    ('sgd', sgd),
    ('perceptron', perceptron),
    ('svc', svc),
    ('linear_svc', linear_svc),
    ('knn', knn)
], voting='hard')


In [74]:
list_of_models = [ridge_adhd, gaussian, decision_tree, 
                  logistic_regression, random_forest, extra_trees, 
                  gradient_boosting, hist_gradient_boosting, sgd, 
                  perceptron, svc, linear_svc, 
                  knn, stacking, voting,
                  adaboost, bagging]
list_of_models_name = ['RidgeClassifier', 'GaussianNB', 'DecisionTreeClassifier', 
                       'LogisticRegression', 'RandomForestClassifier', 'ExtraTreesClassifier',
                       'GradientBoostingClassifier', 'HistGradientBoostingClassifier', 'SGDClassifier',
                        'Perceptron', 'SVC', 'LinearSVC', 
                        'KNeighborsClassifier', 'StackingClassifier', 'VotingClassifier',
                        'AdaBoostClassifier', 'BaggingClassifier']

In [75]:
evaluator = ModelEvaluator(list_of_models, list_of_models_name, 
                           X_train_adhd=X_train_adhd, y_train_adhd=y_train_adhd, X_val_adhd=X_val_adhd, y_val_adhd=y_val_adhd, X_train_sex=X_train_sex, 
                           y_train_sex=y_train_sex, X_val_sex=X_val_sex, y_val_sex=y_val_sex)
df = evaluator.evaluate()
df.sort_values(by=('Sex F', 'F1 Score'), ascending=False)

ADHD_Outcome               Sex F          
                                   Accuracy  F1 Score  Accuracy  F1 Score
StackingClassifier                 0.796680  0.855457  0.810700  0.676056
LogisticRegression                 0.804979  0.858006  0.773663  0.640523
GaussianNB                         0.746888  0.801303  0.736626  0.619048
RidgeClassifier                    0.809129  0.861446  0.757202  0.614379
LinearSVC                          0.809129  0.860606  0.748971  0.611465
VotingClassifier                   0.796680  0.852853  0.777778  0.602941
SVC                                0.788382  0.850440  0.781893  0.601504
GradientBoostingClassifier         0.771784  0.836795  0.773663  0.573643
HistGradientBoostingClassifier     0.759336  0.830409  0.761317  0.560606
SGDClassifier                      0.809129  0.863905  0.658436  0.536313
AdaBoostClassifier                 0.780083  0.838906  0.716049  0.530612
KNeighborsClassifier               0.767635  0.833333  0.654321  0.500000
Perceptron                         0.726141  0.797546  0.674897  0.490323
DecisionTreeClassifier             0.730290  0.797508  0.592593  0.400000
RandomForestClassifier             0.759336  0.831395  0.728395  0.365385
BaggingClassifier                  0.751037  0.822485  0.674897  0.300885
ExtraTreesClassifier               0.746888  0.817910  0.695473  0.260000

In [76]:
list_of_models = [stacking]
list_of_models_name = ['StackingClassifier']

In [77]:
evaluator = ModelEvaluator(list_of_models, list_of_models_name, 
                           X_train_adhd=X_train_adhd, y_train_adhd=y_train_adhd, X_val_adhd=X_val_adhd, y_val_adhd=y_val_adhd, X_train_sex=X_train_sex, 
                           y_train_sex=y_train_sex, X_val_sex=X_val_sex, y_val_sex=y_val_sex)
df = evaluator.evaluate()
df.sort_values(by=('Sex F', 'F1 Score'), ascending=False)

ADHD_Outcome              Sex F          
                       Accuracy  F1 Score Accuracy  F1 Score
StackingClassifier      0.79668  0.855457   0.8107  0.676056

In [78]:
# features_sex_f = features_sex_f + features_connectome_sex
# features_adhd = features_adhd + features_connectome_adhd

# label_adhd = 'ADHD_Outcome'
# label_sex_f = 'Sex_F'

# X_adhd = final_data[features_adhd]
# X_test_adhd = final_data_test[features_adhd]

# #X_sex_f = final_data.drop(columns=['participant_id', 'ADHD_Outcome', 'Sex_F'])

# X_sex_f = final_data[features_sex_f]
# X_test_sex = final_data_test[features_sex_f]
# # X_test_sex = final_data_test.drop(columns=['participant_id'])


# X_adhd = X_adhd.fillna(X_adhd.median())
# X_test_adhd = X_test_adhd.fillna(X_test_adhd.median())
# X_sex_f = X_sex_f.fillna(X_sex_f.median())
# X_test_sex = X_test_sex.fillna(X_test_sex.median())

# # X_adhd_train, X_adhd_test, y_adhd_train, y_adhd_test = train_test_split(X_adhd, y_adhd, test_size=0.2, random_state=42)

# # X_sex_f = final_data[features_sex_f]
# y_adhd_train = final_data[label_adhd]
# y_sex_f_train = final_data[label_sex_f]

# # X_sex_f_train, X_sex_f_test, y_sex_f_train, y_sex_f_test = train_test_split(X_sex_f, y_sex_f, test_size=0.2, random_state=42)

# # Standardize the features
# scaler = StandardScaler()
# X_adhd_train_scaled = scaler.fit_transform(X_adhd)
# X_test_scaled_adhd = scaler.transform(X_test_adhd)
# X_sex_f_train_scaled = scaler.fit_transform(X_sex_f)
# X_test_scaled_sex = scaler.transform(X_test_sex)


# # Train RidgeClassifier for ADHD_Outcome
# ridge_adhd = RidgeClassifier(alpha=1.0)  # Adjust alpha for regularization strength
# ridge_adhd.fit(X_adhd_train_scaled, y_adhd_train)
# y_adhd_pred_train = ridge_adhd.predict(X_adhd_train_scaled)
# y_adhd_pred_test = ridge_adhd.predict(X_test_scaled_adhd)

# # Train RidgeClassifier for Sex_F
# ridge_sex_f = RidgeClassifier(alpha=1.0)
# ridge_sex_f.fit(X_sex_f_train_scaled, y_sex_f_train)
# y_sex_f_pred_train = ridge_sex_f.predict(X_sex_f_train_scaled)
# y_sex_f_pred_test = ridge_sex_f.predict(X_test_scaled_sex)

# # Compute F1-score and accuracy
# f1_adhd_train = f1_score(y_adhd_train, y_adhd_pred_train)
# f1_sex_f_train = f1_score(y_sex_f_train, y_sex_f_pred_train)



# # Print results
# print(f"ADHD_Outcome - F1 Score (Train): {f1_adhd_train}")
# print(f"Sex_F - F1 Score (Train): {f1_sex_f_train}")



# # Save predictions to CSV
# submission = final_data_test[['participant_id']].copy()
# submission['ADHD_Outcome'] = y_adhd_pred_test  # Assuming the first column corresponds to ADHD_Outcome
# submission['Sex_F'] = y_sex_f_pred_test

# # Save results to CSV
# submission.to_csv("submission-1.csv", index=False)
# print("Predictions saved to submission.csv")

In [ ]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import RidgeClassifier, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, accuracy_score
import pandas as pd

# Combine features
features_sex_f = features_sex_f + features_connectome_sex
features_adhd = features_adhd

label_adhd = 'ADHD_Outcome'
label_sex_f = 'Sex_F'

# Prepare data
X_adhd = final_data[features_adhd]
X_test_adhd = final_data_test[features_adhd]

X_sex_f = final_data[features_sex_f]
X_test_sex = final_data_test[features_sex_f]

X_adhd = X_adhd.fillna(X_adhd.median())
X_test_adhd = X_test_adhd.fillna(X_test_adhd.median())
X_sex_f = X_sex_f.fillna(X_sex_f.median())
X_test_sex = X_test_sex.fillna(X_test_sex.median())

y_adhd_train = final_data[label_adhd]
y_sex_f_train = final_data[label_sex_f]

# Standardize the features
scaler = StandardScaler()
X_adhd_train_scaled = scaler.fit_transform(X_adhd)
X_test_scaled_adhd = scaler.transform(X_test_adhd)
X_sex_f_train_scaled = scaler.fit_transform(X_sex_f)
X_test_scaled_sex = scaler.transform(X_test_sex)

# Define base models and stacking classifier, gaussian bayesian, suport vector classifier
base_models = [
    # ('ridge', RidgeClassifier(alpha=1.0)),
    ('logistic', LogisticRegression(max_iter=1000, random_state=42, C=0.1)),
    ('gaussian', GaussianNB()),
    ('svc', SVC(probability=True, random_state=42))
]

stacking_adhd = StackingClassifier(estimators=base_models, final_estimator=LogisticRegression(max_iter=100, random_state=42))
stacking_sex_f = StackingClassifier(estimators=base_models, final_estimator=LogisticRegression(max_iter=100, random_state=42))

# Train StackingClassifier for ADHD_Outcome
stacking_adhd.fit(X_adhd_train_scaled, y_adhd_train)
y_adhd_pred_train = stacking_adhd.predict(X_adhd_train_scaled)
y_adhd_pred_test = stacking_adhd.predict(X_test_scaled_adhd)

# Train StackingClassifier for Sex_F
stacking_sex_f.fit(X_sex_f_train_scaled, y_sex_f_train)
y_sex_f_pred_train = stacking_sex_f.predict(X_sex_f_train_scaled)
y_sex_f_pred_test = stacking_sex_f.predict(X_test_scaled_sex)

# Compute F1-score
f1_adhd_train = f1_score(y_adhd_train, y_adhd_pred_train)
f1_sex_f_train = f1_score(y_sex_f_train, y_sex_f_pred_train)

acc_adhd = accuracy_score(y_adhd_train, y_adhd_pred_train)
acc_sex_f = accuracy_score(y_sex_f_train, y_sex_f_pred_train)

# Print results
print(f"ADHD_Outcome - F1 Score (Train): {f1_adhd_train}")
print(f"Sex_F - F1 Score (Train): {f1_sex_f_train}")

print(f"ADHD_Outcome - Accuracy (Train): {acc_adhd}")
print(f"Sex_F - Accuracy (Train): {acc_sex_f}")

# Save predictions to CSV
submission = final_data_test[['participant_id']].copy()
submission['ADHD_Outcome'] = y_adhd_pred_test
submission['Sex_F'] = y_sex_f_pred_test

submission.to_csv("submission-1.csv", index=False)
print("Predictions saved to submission.csv")

ADHD_Outcome - F1 Score (Train): 0.8618721461187214
Sex_F - F1 Score (Train): 0.5991316931982634
ADHD_Outcome - Accuracy (Train): 0.8004946413849959
Sex_F - Accuracy (Train): 0.7716405605935697
Predictions saved to submission.csv


: 

In [139]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import f1_score

# Define the parameter grid for RandomForestClassifier
param_grid = {
    'n_estimators': [50, 100, 200],  # Number of trees in the forest
    'max_depth': [None, 10, 20, 30],  # Maximum depth of the tree
    'min_samples_split': [2, 5, 10],  # Minimum number of samples required to split an internal node
    'min_samples_leaf': [1, 2, 4],  # Minimum number of samples required to be at a leaf node
    'bootstrap': [True, False]  # Whether bootstrap samples are used when building trees
}

# Initialize the RandomForestClassifier
rf = RandomForestClassifier(random_state=42)

# Use GridSearchCV to find the best hyperparameters
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=3, scoring='f1', verbose=2, n_jobs=-1)

# Fit the model on ADHD data
grid_search.fit(X_adhd_train_scaled, y_adhd_train)

# Get the best parameters
best_params = grid_search.best_params_
print(f"Best parameters for RandomForestClassifier: {best_params}")

# Train the RandomForestClassifier with the best parameters
best_rf_adhd = RandomForestClassifier(**best_params, random_state=42)
best_rf_adhd.fit(X_adhd_train_scaled, y_adhd_train)

# Make predictions
y_adhd_pred_train = best_rf_adhd.predict(X_adhd_train_scaled)
y_adhd_pred_test = best_rf_adhd.predict(X_test_scaled_adhd)

# Compute F1-score
f1_adhd_train = f1_score(y_adhd_train, y_adhd_pred_train)
print(f"ADHD_Outcome - F1 Score (Train): {f1_adhd_train}")

# Repeat the same process for Sex_F
grid_search.fit(X_sex_f_train_scaled, y_sex_f_train)
best_params_sex = grid_search.best_params_
print(f"Best parameters for RandomForestClassifier (Sex_F): {best_params_sex}")

best_rf_sex = RandomForestClassifier(**best_params_sex, random_state=42)
best_rf_sex.fit(X_sex_f_train_scaled, y_sex_f_train)

y_sex_f_pred_train = best_rf_sex.predict(X_sex_f_train_scaled)
y_sex_f_pred_test = best_rf_sex.predict(X_test_scaled_sex)

f1_sex_f_train = f1_score(y_sex_f_train, y_sex_f_pred_train)
print(f"Sex_F - F1 Score (Train): {f1_sex_f_train}")

Fitting 3 folds for each of 216 candidates, totalling 648 fits
Best parameters for RandomForestClassifier: {'bootstrap': True, 'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 50}
ADHD_Outcome - F1 Score (Train): 0.8560523446019629
Fitting 3 folds for each of 216 candidates, totalling 648 fits
Best parameters for RandomForestClassifier (Sex_F): {'bootstrap': False, 'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 50}
Sex_F - F1 Score (Train): 1.0


### Consider

In [80]:
# corr_matrix = data_origin_funct_matrix.drop(columns=['participant_id']).corr()
# # Filter correlations greater than 0.5 (absolute value) 
# strong_correlations = corr_matrix[abs(corr_matrix) > 0.5].dropna(how='all', axis=0).dropna(how='all', axis=1)

# # Display the filtered correlation matrix
# plt.figure(figsize=(10, 6))
# sns.heatmap(strong_correlations, annot=True, fmt=".2f", cmap="coolwarm", linewidths=0.5)
# plt.title("Strong Correlations (> 0.5) Heatmap")

In [81]:
# final_data_cat = pd.merge(data_origin_cate, data_origin_quan, on = 'participant_id')
# final_data = pd.merge(final_data_cat, data_origin_solution, on = 'participant_id')
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.metrics import f1_score

In [82]:
# from xgboost import XGBClassifier
# from sklearn.preprocessing import StandardScaler
# from sklearn.metrics import f1_score

# # Define features and target variables
# features = ['SDQ_SDQ_Hyperactivity', 'SDQ_SDQ_Externalizing', 
#             'SDQ_SDQ_Difficulties_Total', 'SDQ_SDQ_Generating_Impact', 
#             'SDQ_SDQ_Conduct_Problems']

# X_train = final_data[features]
# y_train = final_data[['ADHD_Outcome', 'Sex_F']]
# X_test = final_data_test.drop(columns=['participant_id'])

# # Fill missing values with the median
# X_train = X_train.fillna(X_train.median())
# X_test = X_test.fillna(X_test.median())

# # Standardize the data
# scaler = StandardScaler()
# X_train_scaled = scaler.fit_transform(X_train)
# X_test_scaled = scaler.transform(X_test)

# # Train XGBoost Classifier
# xgb_model = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=6, random_state=42, use_label_encoder=False)
# xgb_model.fit(X_train_scaled, y_train)

# # Make predictions on train and test data
# train_predictions = xgb_model.predict(X_train_scaled)
# test_predictions = xgb_model.predict(X_test_scaled)

# # Compute F1-score for training set with average='weighted' for multilabel classification
# f1_adhd = f1_score(y_train, train_predictions, average='weighted')

# print(f"F1 Score for ADHD_Outcome and Sex_F (Weighted Average): {f1_adhd:.4f}")

# # Create a DataFrame with participant_id and predictions
# submission = final_data_test[['participant_id']].copy()
# submission['ADHD_Outcome'] = test_predictions[:, 0]  # Assuming the first column corresponds to ADHD_Outcome
# submission['Sex_F'] = test_predictions[:, 1]

# # Save results to CSV
# submission.to_csv("submission.csv", index=False)
# print("Predictions saved to submission.csv")

In [83]:
# final_data_mat = pd.merge(data_origin_funct_matrix, data_origin_solution, on = 'participant_id')
# final_data_mat_test = data_test_funct_matrix.copy()

In [84]:
# import pandas as pd
# import numpy as np
# import torch
# import torch.nn.functional as F
# from torch_geometric.data import Data, DataLoader
# from torch_geometric.nn import GCNConv, global_mean_pool
# from sklearn.model_selection import train_test_split
# from sklearn.preprocessing import LabelEncoder, StandardScaler
# from sklearn.metrics import f1_score, accuracy_score
# import umap

# # Step 1: Data Loading and Preparation
# def load_data():
#     # Load datasets
#     connectome_data = pd.read_csv("D:/Datathon/WiDS-Datathon/widsdatathon2025/TRAIN_NEW/TRAIN_FUNCTIONAL_CONNECTOME_MATRICES_new_36P_Pearson.csv")
#     labels = pd.read_excel("D:/Datathon/WiDS-Datathon/widsdatathon2025/TRAIN_NEW/TRAINING_SOLUTIONS.xlsx")
#     new_connectome_data = pd.read_csv("D:/Datathon/WiDS-Datathon/widsdatathon2025/TEST/TEST_FUNCTIONAL_CONNECTOME_MATRICES.csv")

#     # Merge and prepare data
#     data = pd.merge(connectome_data, labels, on="participant_id")
#     X = data.drop(columns=["participant_id", "ADHD_Outcome", "Sex_F"]).values
#     y = data[["ADHD_Outcome", "Sex_F"]]

#     # Encode labels
#     le_adhd = LabelEncoder()
#     le_sex = LabelEncoder()
#     y["ADHD_Outcome"] = le_adhd.fit_transform(y["ADHD_Outcome"])
#     y["Sex_F"] = le_sex.fit_transform(y["Sex_F"])

#     return X, y, new_connectome_data, le_adhd, le_sex

# def apply_umap(X, n_components=128):
#     scaler = StandardScaler()
#     X_scaled = scaler.fit_transform(X)
#     umap_model = umap.UMAP(n_components=n_components, random_state=42)
#     X_umap = umap_model.fit_transform(X_scaled)
#     print(f"UMAP: Reduced to {n_components} dimensions")
#     return X_umap, umap_model

# # Step 2: Graph Data Construction
# def create_graph_data(connectome_vector, label):
#     num_nodes = len(connectome_vector)
#     rows, cols = np.triu_indices(num_nodes)
#     edge_index = torch.tensor([rows, cols], dtype=torch.long)
#     edge_attr = torch.tensor(connectome_vector[rows] * connectome_vector[cols], dtype=torch.float)
#     x = torch.eye(num_nodes, dtype=torch.float)
#     y = torch.tensor(label, dtype=torch.float).view(1, -1)
#     return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y)

# def prepare_graph_data(X, y):
#     return [create_graph_data(X[i], y.iloc[i].values) for i in range(len(X))]

# # Step 3: Enhanced GNN Model
# class BrainGNN(torch.nn.Module):
#     def __init__(self, num_features, num_classes):
#         super().__init__()
#         self.conv1 = GCNConv(num_features, 128)
#         self.bn1 = torch.nn.BatchNorm1d(128)
#         self.conv2 = GCNConv(128, 256)
#         self.bn2 = torch.nn.BatchNorm1d(256)
#         self.conv3 = GCNConv(256, 256)
#         self.dropout = torch.nn.Dropout(0.5)
#         self.fc = torch.nn.Linear(256, num_classes)
        
#     def forward(self, data):
#         x, edge_index, batch = data.x, data.edge_index, data.batch
#         x = F.relu(self.bn1(self.conv1(x, edge_index)))
#         x = F.relu(self.bn2(self.conv2(x, edge_index)))
#         x = F.relu(self.conv3(x, edge_index))
#         x = global_mean_pool(x, batch)
#         x = self.dropout(x)
#         return self.fc(x)

# # Step 4: Training and Evaluation
# def train_epoch(model, loader, optimizer, device):
#     model.train()
#     total_loss = 0
#     criterion = torch.nn.BCEWithLogitsLoss()
#     for data in loader:
#         data = data.to(device)
#         optimizer.zero_grad()
#         out = model(data)
#         loss = criterion(out, data.y.view(-1, 2))
#         loss.backward()
#         optimizer.step()
#         total_loss += loss.item()
#     return total_loss / len(loader)

# def evaluate(model, loader, device):
#     model.eval()
#     preds, targets = [], []
#     criterion = torch.nn.BCEWithLogitsLoss()
#     total_loss = 0
#     with torch.no_grad():
#         for data in loader:
#             data = data.to(device)
#             out = model(data)
#             loss = criterion(out, data.y.view(-1, 2))
#             total_loss += loss.item()
#             preds.append((torch.sigmoid(out) > 0.5).cpu().numpy())
#             targets.append(data.y.view(-1, 2).cpu().numpy())
#     preds = np.concatenate(preds)
#     targets = np.concatenate(targets)
#     f1 = f1_score(targets, preds, average='macro')
#     return total_loss / len(loader), f1

# # Step 5: Complete Training Pipeline
# def train_model(train_loader, val_loader, device):
#     model = BrainGNN(num_features=128, num_classes=2).to(device)
#     optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
#     scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=3)
#     best_f1 = 0
#     for epoch in range(100):
#         train_loss = train_epoch(model, train_loader, optimizer, device)
#         val_loss, val_f1 = evaluate(model, val_loader, device)
#         scheduler.step(val_loss)
#         print(f"Epoch {epoch+1:03d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val F1: {val_f1:.4f}")
#         if val_f1 > best_f1:
#             best_f1 = val_f1
#             torch.save(model.state_dict(), "best_model.pt")
#     model.load_state_dict(torch.load("best_model.pt"))
#     return model

# # Step 6: Prediction and Submission
# def make_predictions(model, new_data, umap_model, le_adhd, le_sex, device):
#     new_X = new_data.drop(columns=["participant_id"]).values
#     new_X_umap = umap_model.transform(new_X)
#     new_graphs = prepare_graph_data(new_X_umap, pd.DataFrame([[0, 0]] * len(new_X_umap)))
#     loader = DataLoader(new_graphs, batch_size=32, shuffle=False)
#     model.eval()
#     predictions = []
#     with torch.no_grad():
#         for data in loader:
#             data = data.to(device)
#             out = model(data)
#             predictions.extend((torch.sigmoid(out) > 0.5).cpu().numpy())
#     submission = pd.DataFrame({
#         "participant_id": new_data["participant_id"],
#         "ADHD_Outcome": le_adhd.inverse_transform([int(p[0]) for p in predictions]),
#         "Sex_F": le_sex.inverse_transform([int(p[1]) for p in predictions])
#     })
#     return submission

# def main():
#     X, y, new_data, le_adhd, le_sex = load_data()
#     X_umap, umap_model = apply_umap(X)
#     graphs = prepare_graph_data(X_umap, y)
#     train_graphs, val_graphs = train_test_split(graphs, test_size=0.2, random_state=42)
#     train_loader = DataLoader(train_graphs, batch_size=32, shuffle=True)
#     val_loader = DataLoader(val_graphs, batch_size=32, shuffle=False)
#     device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#     model = train_model(train_loader, val_loader, device)
#     submission = make_predictions(model, new_data, umap_model, le_adhd, le_sex, device)
#     submission.to_csv("submission.csv", index=False)
#     print("Submission saved to submission.csv")

# if __name__ == "__main__":
#     main()